# Read

In [ ]:
from pathlib import Path
import re
import gc
import time
import pandas as pd
from openpyxl import load_workbook

# ============================================================
# READ ONLY JOB FILES 2010-2025
# MEMORY OPTIMIZED VERSION WITH VISIBLE LOADING
#
# NO SAVE
# NO CSV
# NO EXCEL OUTPUT
# NO JOIN
# NO MERGE
# NO CONCAT
#
# FIX INCLUDED:
# Some files have different column counts.
# Example:
# header = 38 columns
# data row = 29 columns
#
# This code pads short rows with None
# and cuts long rows so preview will not crash.
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
)

FOLDERS = [
    BASE_DIR / "2010-2011 all combine no need join",
    BASE_DIR / "2012-2018",
    BASE_DIR / "2019-2025",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def now_time():
    return time.strftime("%H:%M:%S")


def get_year_from_filename(file_path):
    match = re.search(r"(20\d{2})", file_path.name)
    if match:
        return int(match.group(1))
    return None


def choose_data_sheet(sheet_names):
    """
    Pick real data sheet.
    Skip description/update/filler sheets.
    """
    skip_words = ["field", "description", "update", "filler"]

    for sheet in sheet_names:
        low = sheet.lower()
        if not any(word in low for word in skip_words):
            return sheet

    return sheet_names[0]


def print_progress(current, total, file_name, status):
    percent = (current / total) * 100 if total else 0
    bar_length = 30
    filled = int(bar_length * current / total) if total else 0
    bar = "█" * filled + "-" * (bar_length - filled)

    print(
        f"[{now_time()}] [{bar}] {percent:6.2f}% "
        f"({current}/{total}) | {status} | {file_name}",
        flush=True
    )


def make_unique_columns(columns):
    """
    Make column names safe:
    - blank column names become BLANK_COLUMN_#
    - duplicate column names get _2, _3, etc.
    """
    clean_cols = []
    seen = {}

    for idx, col in enumerate(columns, start=1):
        if col is None:
            col = ""

        col = str(col).strip()

        if col == "":
            col = f"BLANK_COLUMN_{idx}"

        if col in seen:
            seen[col] += 1
            col = f"{col}_{seen[col]}"
        else:
            seen[col] = 1

        clean_cols.append(col)

    return clean_cols


def fix_row_length(row, target_len):
    """
    Fix mismatch:
    - If row is shorter than header, pad with None
    - If row is longer than header, cut extra values
    """
    row = list(row)

    if len(row) < target_len:
        row = row + [None] * (target_len - len(row))

    elif len(row) > target_len:
        row = row[:target_len]

    return row


def read_excel_preview_optimized(file_path):
    """
    MEMORY OPTIMIZED:
    Opens workbook in read_only mode.
    Does not load whole file.
    Only reads:
    - sheet names
    - row count
    - column count
    - header row
    - first 5 data rows

    SAFE FOR DIFFERENT COLUMN COUNTS:
    Header and preview rows are forced to same length.
    """

    print(f"    Opening workbook...", flush=True)

    wb = None

    try:
        wb = load_workbook(
            filename=file_path,
            read_only=True,
            data_only=True
        )

        print(f"    Workbook opened.", flush=True)

        sheet_names = wb.sheetnames
        sheet_name = choose_data_sheet(sheet_names)
        ws = wb[sheet_name]

        print(f"    Using sheet: {sheet_name}", flush=True)

        total_rows = ws.max_row or 0
        total_cols = ws.max_column or 0
        data_rows = max(total_rows - 1, 0)

        print(f"    Reading header + first 5 rows only...", flush=True)

        rows = []
        for row in ws.iter_rows(min_row=1, max_row=6, values_only=True):
            rows.append(row)

    finally:
        if wb is not None:
            wb.close()
            print(f"    Workbook closed.", flush=True)

    if len(rows) == 0:
        columns = []
        preview_df = pd.DataFrame()
        mismatch_note = "EMPTY SHEET"
        header_col_count = 0
        widest_preview_row = 0

    else:
        header_raw = list(rows[0])
        data_raw = rows[1:]

        header_col_count = len(header_raw)
        widest_preview_row = max([len(r) for r in data_raw], default=0)

        # Use the biggest count between worksheet max_column, header, and preview rows
        target_cols = max(total_cols, header_col_count, widest_preview_row)

        # Fix header length
        header_fixed = fix_row_length(header_raw, target_cols)
        columns = make_unique_columns(header_fixed)

        # Fix each preview row length
        fixed_data = []
        short_rows = 0
        long_rows = 0

        for row in data_raw:
            original_len = len(row)

            if original_len < target_cols:
                short_rows += 1
            elif original_len > target_cols:
                long_rows += 1

            fixed_data.append(fix_row_length(row, target_cols))

        preview_df = pd.DataFrame(fixed_data, columns=columns)

        if short_rows > 0 or long_rows > 0:
            mismatch_note = (
                f"FIXED COLUMN MISMATCH | "
                f"header={header_col_count}, "
                f"widest_preview_row={widest_preview_row}, "
                f"worksheet_max_col={total_cols}, "
                f"target={target_cols}, "
                f"short_rows_padded={short_rows}, "
                f"long_rows_cut={long_rows}"
            )
        else:
            mismatch_note = "OK"

    return {
        "sheet_names": sheet_names,
        "sheet_name": sheet_name,
        "rows": data_rows,
        "columns": total_cols,
        "column_names": columns,
        "preview_df": preview_df,
        "mismatch_note": mismatch_note,
        "header_col_count": header_col_count,
        "widest_preview_row": widest_preview_row,
    }


# ============================================================
# FIND FILES
# ============================================================

all_files = []

print("=" * 80, flush=True)
print("SEARCHING FOR FILES", flush=True)
print("=" * 80, flush=True)

for folder in FOLDERS:
    print(f"Checking folder: {folder}", flush=True)

    if not folder.exists():
        print(f"❌ Folder not found: {folder}", flush=True)
        continue

    files = sorted([
        f for f in folder.glob("*.xlsx")
        if not f.name.startswith("~$")
    ])

    for file_path in files:
        year = get_year_from_filename(file_path)

        all_files.append({
            "year": year,
            "folder": folder.name,
            "file_path": file_path,
        })

all_files = sorted(
    all_files,
    key=lambda x: (x["year"] if x["year"] else 9999, x["file_path"].name)
)

print("\n" + "=" * 80, flush=True)
print("FILES FOUND", flush=True)
print("=" * 80, flush=True)

for item in all_files:
    print(f"{item['year']} | {item['folder']} | {item['file_path'].name}", flush=True)

print(f"\nTotal files found: {len(all_files)}", flush=True)


# ============================================================
# READ ONLY INSPECTION
# ============================================================

summary_rows = []

print("\n" + "=" * 80, flush=True)
print("OPTIMIZATION STATUS", flush=True)
print("=" * 80, flush=True)
print("Memory optimization: ON", flush=True)
print("Full file loading: OFF", flush=True)
print("Saving files: OFF", flush=True)
print("Combine/concat: OFF", flush=True)
print("Preview rows per file: 5", flush=True)
print("One file open at a time: ON", flush=True)
print("Visible loading: ON", flush=True)
print("Column mismatch fix: ON", flush=True)

total_files = len(all_files)
overall_start = time.time()

for i, item in enumerate(all_files, start=1):
    file_start = time.time()

    file_path = item["file_path"]
    year = item["year"]

    print("\n" + "=" * 80, flush=True)
    print_progress(i, total_files, file_path.name, "STARTING")
    print(f"YEAR: {year}", flush=True)
    print(f"FOLDER: {item['folder']}", flush=True)
    print(f"FILE: {file_path.name}", flush=True)
    print("=" * 80, flush=True)

    file_status = "OK"

    try:
        info = read_excel_preview_optimized(file_path)

        print(f"\nSheets: {info['sheet_names']}", flush=True)
        print(f"Using sheet: {info['sheet_name']}", flush=True)
        print(f"Rows: {info['rows']:,}", flush=True)
        print(f"Columns from worksheet: {info['columns']:,}", flush=True)
        print(f"Header columns read: {info['header_col_count']:,}", flush=True)
        print(f"Widest preview row: {info['widest_preview_row']:,}", flush=True)
        print(f"Mismatch note: {info['mismatch_note']}", flush=True)

        print("\nColumn names:", flush=True)
        for col_num, col_name in enumerate(info["column_names"], start=1):
            print(f"{col_num}. {col_name}", flush=True)

        print("\nFirst 5 rows:", flush=True)
        print(info["preview_df"], flush=True)

        summary_rows.append({
            "year": year,
            "source_folder": item["folder"],
            "source_file": file_path.name,
            "source_sheet": info["sheet_name"],
            "rows": info["rows"],
            "columns_from_worksheet": info["columns"],
            "header_columns_read": info["header_col_count"],
            "widest_preview_row": info["widest_preview_row"],
            "mismatch_note": info["mismatch_note"],
            "status": "OK",
            "seconds": round(time.time() - file_start, 2),
        })

        del info

    except Exception as e:
        file_status = "ERROR"

        print(f"❌ ERROR reading file: {file_path.name}", flush=True)
        print(f"Error message: {e}", flush=True)

        summary_rows.append({
            "year": year,
            "source_folder": item["folder"],
            "source_file": file_path.name,
            "source_sheet": None,
            "rows": None,
            "columns_from_worksheet": None,
            "header_columns_read": None,
            "widest_preview_row": None,
            "mismatch_note": str(e),
            "status": "ERROR",
            "seconds": round(time.time() - file_start, 2),
        })

    gc.collect()

    print_progress(i, total_files, file_path.name, file_status)
    print(f"Time for this file: {round(time.time() - file_start, 2)} seconds", flush=True)


# ============================================================
# SMALL SUMMARY ONLY
# STILL NO SAVE
# ============================================================

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 80, flush=True)
print("READ ONLY SUMMARY", flush=True)
print("=" * 80, flush=True)
print(summary_df, flush=True)

print("\n" + "=" * 80, flush=True)
print("FINAL STATUS", flush=True)
print("=" * 80, flush=True)
print(f"Total files checked: {len(summary_df)}", flush=True)
print(f"Total time: {round(time.time() - overall_start, 2)} seconds", flush=True)
print("DONE. Nothing was saved.", flush=True)

# Save

In [ ]:
from pathlib import Path
import re
import csv
import gc
import time
from openpyxl import load_workbook

# ============================================================
# MERGE JOB FILES 2010-2025 INTO ONE CSV
# MEMORY OPTIMIZED VERSION
#
# OUTPUT:
# ~/Downloads/all_2010-2025_job.csv
#
# MEMORY OPTIMIZATION:
# 1. Opens one Excel file at a time
# 2. Uses openpyxl read_only=True
# 3. Streams row by row
# 4. Does NOT load full files into pandas
# 5. Does NOT concat
# 6. Writes directly to one CSV
#
# FIXES:
# 1. Different column names by year
# 2. Upper/lowercase column names
# 3. Spaces in column names
# 4. 2010 / 2011 column mismatch
# 5. Short rows padded
# 6. Long rows cut
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
)

FOLDERS = [
    BASE_DIR / "2010-2011 all combine no need join",
    BASE_DIR / "2012-2018",
    BASE_DIR / "2019-2025",
]

OUTPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job.csv"

# If True, deletes old output file before starting
OVERWRITE_OUTPUT = True

# Print loading update every N rows
PRINT_EVERY_ROWS = 50_000


# ============================================================
# FINAL OUTPUT COLUMNS
# ============================================================

FINAL_COLUMNS = [
    "year",
    "release",
    "source_workbook",
    "source_folder",
    "source_sheet",
    "source_period",
    "source_file",
    "source_path",

    "area",
    "area_title",
    "area_type",
    "prim_state",
    "state",
    "state_abbr",

    "naics",
    "naics_title",
    "i_group",
    "own_code",
    "ownership",

    "occ_code",
    "occ_title",
    "o_group",
    "group",
    "occ_group",

    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_quotient",
    "pct_total",
    "pct_rpt",

    "h_mean",
    "a_mean",
    "mean_prse",

    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",

    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",

    "annual",
    "hourly",

    "data_level",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def now_time():
    return time.strftime("%H:%M:%S")


def get_year_from_filename(file_path):
    match = re.search(r"(20\d{2})", file_path.name)
    if match:
        return int(match.group(1))
    return None


def choose_data_sheet(sheet_names):
    """
    Pick real data sheet.
    Skip description/update/filler sheets.
    """
    skip_words = ["field", "description", "update", "filler"]

    for sheet in sheet_names:
        low = sheet.lower()
        if not any(word in low for word in skip_words):
            return sheet

    return sheet_names[0]


def print_progress(current, total, file_name, status):
    percent = (current / total) * 100 if total else 0
    bar_length = 30
    filled = int(bar_length * current / total) if total else 0
    bar = "█" * filled + "-" * (bar_length - filled)

    print(
        f"[{now_time()}] [{bar}] {percent:6.2f}% "
        f"({current}/{total}) | {status} | {file_name}",
        flush=True
    )


def clean_column_name(col):
    """
    Standardize column name:
    - lowercase
    - strip spaces
    - replace spaces with underscore
    - normalize common weird names
    """
    if col is None:
        return ""

    col = str(col).strip().lower()
    col = col.replace("\n", " ")
    col = re.sub(r"\s+", " ", col)
    col = col.replace(" ", "_")

    # Remove duplicate underscores
    col = re.sub(r"_+", "_", col)

    return col


def make_unique_columns(columns):
    """
    Make column names safe:
    - blank column names become blank_column_#
    - duplicate names get _2, _3, etc.
    """
    clean_cols = []
    seen = {}

    for idx, col in enumerate(columns, start=1):
        col = clean_column_name(col)

        if col == "":
            col = f"blank_column_{idx}"

        if col in seen:
            seen[col] += 1
            col = f"{col}_{seen[col]}"
        else:
            seen[col] = 1

        clean_cols.append(col)

    return clean_cols


def fix_row_length(row, target_len):
    """
    Fix mismatch:
    - If row is shorter than header, pad with None
    - If row is longer than header, cut extra values
    """
    row = list(row)

    if len(row) < target_len:
        row = row + [None] * (target_len - len(row))

    elif len(row) > target_len:
        row = row[:target_len]

    return row


def normalize_key(col):
    """
    Map different year column names into final standard names.
    """
    col = clean_column_name(col)

    mapping = {
        # Area
        "area": "area",
        "area_name": "area_title",
        "area_title": "area_title",
        "area_type": "area_type",

        # State
        "prim_state": "prim_state",
        "state": "state",
        "st": "state_abbr",

        # Industry
        "naics": "naics",
        "naics_title": "naics_title",
        "i_group": "i_group",

        # Ownership
        "own_code": "own_code",
        "ownership": "ownership",

        # Occupation
        "occ_code": "occ_code",
        "occ_code_": "occ_code",
        "occ_title": "occ_title",
        "occ_title_": "occ_title",
        "o_group": "o_group",
        "group": "group",
        "occ_group": "occ_group",

        # Employment
        "tot_emp": "tot_emp",
        "emp_prse": "emp_prse",
        "jobs_1000": "jobs_1000",
        "jobs_1000_orig": "jobs_1000",

        # Location quotient
        "loc_q": "loc_quotient",
        "loc_quotient": "loc_quotient",
        "loc_quotient_": "loc_quotient",

        # Percent total
        "pct_tot": "pct_total",
        "pct_total": "pct_total",
        "pct_rpt": "pct_rpt",

        # Wages
        "h_mean": "h_mean",
        "a_mean": "a_mean",
        "mean_prse": "mean_prse",

        "h_pct10": "h_pct10",
        "h_pct25": "h_pct25",
        "h_median": "h_median",
        "h_pct75": "h_pct75",
        "h_pct90": "h_pct90",

        "a_pct10": "a_pct10",
        "a_pct25": "a_pct25",
        "a_median": "a_median",
        "a_pct75": "a_pct75",
        "a_pct90": "a_pct90",

        "annual": "annual",
        "hourly": "hourly",

        # Source/meta
        "year": "year",
        "release": "release",
        "data_level": "data_level",
        "source_folder": "source_folder",
        "source_file": "source_file",
        "source_path": "source_path",
        "source_sheet": "source_sheet",
        "source_period": "source_period",
    }

    return mapping.get(col, col)


def find_files():
    all_files = []

    print("=" * 80, flush=True)
    print("SEARCHING FOR FILES", flush=True)
    print("=" * 80, flush=True)

    for folder in FOLDERS:
        print(f"Checking folder: {folder}", flush=True)

        if not folder.exists():
            print(f"❌ Folder not found: {folder}", flush=True)
            continue

        files = sorted([
            f for f in folder.glob("*.xlsx")
            if not f.name.startswith("~$")
        ])

        for file_path in files:
            year = get_year_from_filename(file_path)

            all_files.append({
                "year": year,
                "folder": folder.name,
                "file_path": file_path,
            })

    all_files = sorted(
        all_files,
        key=lambda x: (x["year"] if x["year"] else 9999, x["file_path"].name)
    )

    print("\n" + "=" * 80, flush=True)
    print("FILES FOUND", flush=True)
    print("=" * 80, flush=True)

    for item in all_files:
        print(f"{item['year']} | {item['folder']} | {item['file_path'].name}", flush=True)

    print(f"\nTotal files found: {len(all_files)}", flush=True)

    return all_files


def row_to_final_dict(row_dict, year, folder_name, workbook_name, sheet_name):
    """
    Convert one source row into final output format.
    """
    out = {col: "" for col in FINAL_COLUMNS}

    # Default metadata
    out["year"] = year
    out["source_workbook"] = workbook_name
    out["source_folder"] = folder_name
    out["source_sheet"] = sheet_name

    # Fill values from original row
    for old_col, value in row_dict.items():
        new_col = normalize_key(old_col)

        if new_col in out:
            if value is None:
                value = ""
            out[new_col] = value

    # If year was blank in file, keep filename year
    if out["year"] in ["", None]:
        out["year"] = year

    # If source_file does not exist in that year, use workbook name
    if out["source_file"] in ["", None]:
        out["source_file"] = workbook_name

    # If source_folder does not exist in that year, use folder name
    if out["source_folder"] in ["", None]:
        out["source_folder"] = folder_name

    # If source_sheet does not exist in that year, use chosen sheet name
    if out["source_sheet"] in ["", None]:
        out["source_sheet"] = sheet_name

    return out


def merge_one_excel_file(item, writer, file_index, total_files):
    """
    Stream one Excel workbook into the output CSV.
    """
    file_path = item["file_path"]
    year = item["year"]
    folder_name = item["folder"]
    workbook_name = file_path.name

    wb = None
    rows_written = 0
    file_start = time.time()

    print("\n" + "=" * 80, flush=True)
    print_progress(file_index, total_files, workbook_name, "STARTING")
    print(f"YEAR: {year}", flush=True)
    print(f"FOLDER: {folder_name}", flush=True)
    print(f"FILE: {workbook_name}", flush=True)
    print("=" * 80, flush=True)

    try:
        print("    Opening workbook...", flush=True)

        wb = load_workbook(
            filename=file_path,
            read_only=True,
            data_only=True
        )

        sheet_name = choose_data_sheet(wb.sheetnames)
        ws = wb[sheet_name]

        print(f"    Workbook opened.", flush=True)
        print(f"    Using sheet: {sheet_name}", flush=True)
        print("    Reading and writing rows...", flush=True)

        row_iter = ws.iter_rows(values_only=True)

        try:
            header_raw = next(row_iter)
        except StopIteration:
            print("    EMPTY SHEET - skipped.", flush=True)
            return 0

        header_len = len(header_raw)
        columns = make_unique_columns(header_raw)

        for row_num, row in enumerate(row_iter, start=2):
            # Fix row/header mismatch safely
            target_len = max(header_len, len(row))
            header_fixed = fix_row_length(columns, target_len)
            row_fixed = fix_row_length(row, target_len)

            row_dict = dict(zip(header_fixed, row_fixed))

            final_row = row_to_final_dict(
                row_dict=row_dict,
                year=year,
                folder_name=folder_name,
                workbook_name=workbook_name,
                sheet_name=sheet_name,
            )

            writer.writerow(final_row)
            rows_written += 1

            if rows_written % PRINT_EVERY_ROWS == 0:
                elapsed = time.time() - file_start
                print(
                    f"    {rows_written:,} rows written from {workbook_name} "
                    f"| elapsed {elapsed:.1f} sec",
                    flush=True
                )

        print(
            f"    DONE file: {workbook_name} | rows written: {rows_written:,}",
            flush=True
        )

        return rows_written

    finally:
        if wb is not None:
            wb.close()
            print("    Workbook closed.", flush=True)

        gc.collect()


# ============================================================
# MAIN MERGE
# ============================================================

print("=" * 80, flush=True)
print("MERGE JOB FILES 2010-2025", flush=True)
print("=" * 80, flush=True)
print("Memory optimization: ON", flush=True)
print("One workbook open at a time: ON", flush=True)
print("Full file loading: OFF", flush=True)
print("Pandas concat: OFF", flush=True)
print("Output format: CSV", flush=True)
print(f"Output file: {OUTPUT_FILE}", flush=True)

all_files = find_files()

if OVERWRITE_OUTPUT and OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()
    print(f"\nOld output deleted: {OUTPUT_FILE}", flush=True)

overall_start = time.time()
total_rows_written = 0
summary_rows = []

with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=FINAL_COLUMNS,
        extrasaction="ignore"
    )

    writer.writeheader()

    total_files = len(all_files)

    for i, item in enumerate(all_files, start=1):
        file_start = time.time()

        try:
            rows_written = merge_one_excel_file(
                item=item,
                writer=writer,
                file_index=i,
                total_files=total_files
            )

            status = "OK"

        except Exception as e:
            rows_written = 0
            status = "ERROR"

            print(f"❌ ERROR merging file: {item['file_path'].name}", flush=True)
            print(f"Error message: {e}", flush=True)

        seconds = round(time.time() - file_start, 2)
        total_rows_written += rows_written

        summary_rows.append({
            "year": item["year"],
            "folder": item["folder"],
            "file": item["file_path"].name,
            "rows_written": rows_written,
            "status": status,
            "seconds": seconds,
        })

        print_progress(i, total_files, item["file_path"].name, status)
        print(f"Time for this file: {seconds} seconds", flush=True)
        print(f"Total rows written so far: {total_rows_written:,}", flush=True)

        gc.collect()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80, flush=True)
print("MERGE SUMMARY", flush=True)
print("=" * 80, flush=True)

for row in summary_rows:
    print(
        f"{row['year']} | {row['status']} | "
        f"{row['rows_written']:,} rows | "
        f"{row['seconds']} sec | {row['file']}",
        flush=True
    )

print("\n" + "=" * 80, flush=True)
print("FINAL STATUS", flush=True)
print("=" * 80, flush=True)
print(f"Output file: {OUTPUT_FILE}", flush=True)
print(f"Total files processed: {len(summary_rows)}", flush=True)
print(f"Total rows written: {total_rows_written:,}", flush=True)
print(f"Total time: {round(time.time() - overall_start, 2)} seconds", flush=True)
print("DONE.", flush=True)

# Pre cleanning - scan 1

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc

# ============================================================
# READ ONLY CSV INSPECTION
# MEMORY OPTIMIZED
#
# FILE:
# ~/Downloads/all_2010-2025_job.csv
#
# DOES:
# 1. Reads header only first
# 2. Reads file in chunks
# 3. Prints loading progress
# 4. Counts rows and columns
# 5. Checks missing / blank / dirty values
# 6. Checks duplicate-looking rows using row hash
# 7. Does NOT clean
# 8. Does NOT save
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

FILE = Path.home() / "Downloads" / "all_2010-2025_job.csv"

CHUNKSIZE = 100_000

# How many unique examples to keep per column
MAX_EXAMPLES_PER_COL = 10

# Values that often mean missing / dirty data in BLS-style job files
NA_LIKE_VALUES = {
    "na", "n/a", "nan", "null", "none",
    "#n/a", "#na",
    "-", "--",
    "*", "**", "***",
    ""
}

# ============================================================
# SAFETY CHECK
# ============================================================

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("=" * 90)
print("READ ONLY CSV INSPECTION")
print("=" * 90)
print(f"File: {FILE}")
print(f"File size: {FILE.stat().st_size / (1024 ** 2):,.2f} MB")
print(f"Chunksize: {CHUNKSIZE:,}")
print()

# ============================================================
# FIND ENCODING
# ============================================================

encoding_to_try = ["utf-8", "utf-8-sig", "latin1"]
ENCODING = None

for enc in encoding_to_try:
    try:
        test_header = pd.read_csv(FILE, nrows=0, encoding=enc)
        ENCODING = enc
        break
    except UnicodeDecodeError:
        pass

if ENCODING is None:
    raise UnicodeDecodeError("Could not read file with utf-8, utf-8-sig, or latin1")

print(f"Encoding used: {ENCODING}")
print()

# ============================================================
# READ HEADER ONLY
# ============================================================

header_df = pd.read_csv(FILE, nrows=0, encoding=ENCODING)
columns = list(header_df.columns)

print("=" * 90)
print("COLUMN CHECK")
print("=" * 90)
print(f"Total columns: {len(columns):,}")
print()

for i, col in enumerate(columns, start=1):
    print(f"{i:>3}. {col}")

print()

# ============================================================
# BASIC COLUMN NAME CLEANING WARNINGS
# Does not clean, only warns
# ============================================================

column_name_issues = []

for col in columns:
    issues = []

    if col != col.strip():
        issues.append("leading/trailing space in column name")

    if " " in col:
        issues.append("space inside column name")

    if col == "":
        issues.append("blank column name")

    if col.lower() != col:
        issues.append("mixed/upper case column name")

    if issues:
        column_name_issues.append({
            "column": col,
            "issue": "; ".join(issues)
        })

print("=" * 90)
print("COLUMN NAME ISSUES")
print("=" * 90)

if column_name_issues:
    column_name_issues_df = pd.DataFrame(column_name_issues)
    display(column_name_issues_df)
else:
    print("No obvious column name issues found.")

print()

# ============================================================
# PREVIEW FIRST ROWS
# ============================================================

print("=" * 90)
print("FIRST 10 ROWS PREVIEW")
print("=" * 90)

preview_df = pd.read_csv(
    FILE,
    nrows=10,
    encoding=ENCODING,
    dtype="string",
    keep_default_na=False
)

display(preview_df)

print()

# ============================================================
# CHUNK INSPECTION
# ============================================================

start_time = time.time()

total_rows = 0
chunk_count = 0

blank_counts = pd.Series(0, index=columns, dtype="int64")
na_like_counts = pd.Series(0, index=columns, dtype="int64")
whitespace_counts = pd.Series(0, index=columns, dtype="int64")
nonblank_counts = pd.Series(0, index=columns, dtype="int64")

numeric_possible_counts = pd.Series(0, index=columns, dtype="int64")
numeric_bad_counts = pd.Series(0, index=columns, dtype="int64")

example_values = {col: [] for col in columns}

row_hash_seen = set()
duplicate_hash_count = 0

file_size = FILE.stat().st_size
bytes_read_estimate = 0

print("=" * 90)
print("SCANNING FILE IN CHUNKS")
print("=" * 90)

reader = pd.read_csv(
    FILE,
    chunksize=CHUNKSIZE,
    encoding=ENCODING,
    dtype="string",
    keep_default_na=False,
    low_memory=False
)

for chunk in reader:
    chunk_count += 1
    chunk_rows = len(chunk)
    total_rows += chunk_rows

    # Estimate progress by memory usage of rows, not perfect but useful
    elapsed = time.time() - start_time

    # Make sure same columns exist
    if list(chunk.columns) != columns:
        print("WARNING: Column mismatch found in chunk:")
        print(f"Chunk number: {chunk_count}")
        print("Expected columns:")
        print(columns)
        print("Actual columns:")
        print(list(chunk.columns))

    # Duplicate-looking rows by hash
    row_hashes = pd.util.hash_pandas_object(chunk, index=False).values

    for h in row_hashes:
        if h in row_hash_seen:
            duplicate_hash_count += 1
        else:
            row_hash_seen.add(h)

    # Per-column checks
    for col in columns:
        s = chunk[col].astype("string")

        stripped = s.str.strip()

        blank_mask = stripped.eq("")
        blank_counts[col] += int(blank_mask.sum())

        nonblank_mask = ~blank_mask
        nonblank_counts[col] += int(nonblank_mask.sum())

        # NA-like dirty values
        lower_s = stripped.str.lower()
        na_like_mask = lower_s.isin(NA_LIKE_VALUES)
        na_like_counts[col] += int(na_like_mask.sum())

        # Leading/trailing whitespace inside values
        whitespace_mask = s.ne(stripped) & nonblank_mask
        whitespace_counts[col] += int(whitespace_mask.sum())

        # Numeric-looking check
        # Remove commas, dollar signs, and percent signs only for checking
        cleaned_numeric_test = (
            stripped
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric_converted = pd.to_numeric(cleaned_numeric_test[nonblank_mask], errors="coerce")

        numeric_possible_counts[col] += int(numeric_converted.notna().sum())
        numeric_bad_counts[col] += int(numeric_converted.isna().sum())

        # Keep a few example values
        if len(example_values[col]) < MAX_EXAMPLES_PER_COL:
            unique_vals = stripped[nonblank_mask].drop_duplicates().head(MAX_EXAMPLES_PER_COL).tolist()
            for val in unique_vals:
                if val not in example_values[col] and len(example_values[col]) < MAX_EXAMPLES_PER_COL:
                    example_values[col].append(val)

    percent_done = "unknown"
    rows_per_sec = total_rows / elapsed if elapsed > 0 else 0

    print(
        f"Chunk {chunk_count:,} done | "
        f"Rows read: {total_rows:,} | "
        f"Speed: {rows_per_sec:,.0f} rows/sec | "
        f"Time: {elapsed:,.1f} sec",
        flush=True
    )

    del chunk
    gc.collect()

end_time = time.time()
total_time = end_time - start_time

print()
print("=" * 90)
print("SCAN COMPLETE")
print("=" * 90)
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(columns):,}")
print(f"Chunks scanned: {chunk_count:,}")
print(f"Duplicate-looking rows by row hash: {duplicate_hash_count:,}")
print(f"Total time: {total_time:,.2f} seconds")
print()

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({
    "column": columns,
    "nonblank_count": [int(nonblank_counts[col]) for col in columns],
    "blank_count": [int(blank_counts[col]) for col in columns],
    "na_like_count": [int(na_like_counts[col]) for col in columns],
    "whitespace_value_count": [int(whitespace_counts[col]) for col in columns],
    "numeric_possible_count": [int(numeric_possible_counts[col]) for col in columns],
    "numeric_bad_count": [int(numeric_bad_counts[col]) for col in columns],
})

summary_df["total_rows"] = total_rows

summary_df["blank_percent"] = (
    summary_df["blank_count"] / summary_df["total_rows"] * 100
).round(2)

summary_df["na_like_percent"] = (
    summary_df["na_like_count"] / summary_df["total_rows"] * 100
).round(2)

summary_df["numeric_possible_percent_of_nonblank"] = np.where(
    summary_df["nonblank_count"] > 0,
    summary_df["numeric_possible_count"] / summary_df["nonblank_count"] * 100,
    0
).round(2)

summary_df["example_values"] = summary_df["column"].map(
    lambda col: " | ".join(example_values[col])
)

# Put most suspicious columns first
summary_df = summary_df.sort_values(
    by=[
        "blank_count",
        "na_like_count",
        "whitespace_value_count",
        "numeric_bad_count"
    ],
    ascending=False
).reset_index(drop=True)

print("=" * 90)
print("COLUMN CLEANING CHECK SUMMARY")
print("=" * 90)
display(summary_df)

# ============================================================
# POSSIBLE CLEANING FLAGS
# ============================================================

flags = []

for _, row in summary_df.iterrows():
    col = row["column"]
    issue_list = []

    if row["blank_count"] > 0:
        issue_list.append(f"has blanks: {int(row['blank_count']):,}")

    if row["na_like_count"] > 0:
        issue_list.append(f"has NA-like values: {int(row['na_like_count']):,}")

    if row["whitespace_value_count"] > 0:
        issue_list.append(f"has leading/trailing spaces in values: {int(row['whitespace_value_count']):,}")

    # If most nonblank values are numeric but some are not, this column may need numeric cleaning
    if (
        row["nonblank_count"] > 0
        and row["numeric_possible_percent_of_nonblank"] >= 70
        and row["numeric_bad_count"] > 0
    ):
        issue_list.append(
            f"mostly numeric but has non-numeric values: {int(row['numeric_bad_count']):,}"
        )

    if issue_list:
        flags.append({
            "column": col,
            "possible_issue": "; ".join(issue_list),
            "example_values": row["example_values"]
        })

flags_df = pd.DataFrame(flags)

print()
print("=" * 90)
print("POSSIBLE COLUMNS THAT MAY NEED CLEANING")
print("=" * 90)

if len(flags_df) > 0:
    display(flags_df)
else:
    print("No obvious cleaning issues found.")

print()

# ============================================================
# FINAL DECISION HELP
# ============================================================

print("=" * 90)
print("FINAL QUICK ANSWER")
print("=" * 90)

print(f"Rows: {total_rows:,}")
print(f"Columns: {len(columns):,}")
print(f"Duplicate-looking rows: {duplicate_hash_count:,}")

if len(flags_df) > 0:
    print()
    print("This file may need cleaning before analysis.")
    print("Check the table above for columns with blanks, NA-like values, spaces, or mixed numeric/text values.")
else:
    print()
    print("No major cleaning issue was found from this scan.")

print()
print("Important: This code only inspects the file. It does not clean or save anything.")

# Pre cleanning - scan 2

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import time
import gc

# ============================================================
# JOB CSV DECISION SCAN ONLY
# MEMORY OPTIMIZED
#
# FILE:
# ~/Downloads/all_2010-2025_job.csv
#
# DOES:
# 1. READ ONLY
# 2. NO CLEANING
# 3. NO SAVE
# 4. NO CSV OUTPUT
# 5. NO EXCEL OUTPUT
# 6. PRINT ANALYSIS TABLES ONLY
#
# GOAL:
# Decide if cleaning is needed before analysis.
#
# MEMORY OPTIMIZATION:
# - Reads CSV in chunks
# - Stores only summary counters
# - Does not load full file into pandas
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

FILE = Path.home() / "Downloads" / "all_2010-2025_job.csv"

CHUNKSIZE = 100_000

# Duplicate check can use more memory because it must remember key hashes.
# Keep True if you want duplicate decision.
CHECK_DUPLICATE_KEYS = True

# These are the main columns used for analysis/math
NUMERIC_ANALYSIS_COLS = [
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_quotient",
    "pct_total",
    "pct_rpt",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
]

# Important ID/category columns
TEXT_ANALYSIS_COLS = [
    "year",
    "area",
    "area_title",
    "state",
    "state_abbr",
    "naics",
    "naics_title",
    "own_code",
    "ownership",
    "occ_code",
    "occ_title",
    "group",
    "o_group",
    "occ_group",
    "i_group",
    "data_level",
    "source_workbook",
    "source_file",
    "source_sheet",
    "source_period",
]

# Key columns for duplicate decision
# This checks whether same job/industry/area/year appears more than once.
DUPLICATE_KEY_COLS = [
    "year",
    "area",
    "naics",
    "own_code",
    "occ_code",
    "data_level",
]

# Dirty symbols often found in BLS/OES data
DIRTY_SYMBOLS = ["#", "*", "**", "***", "~", "-", "--"]

NA_LIKE_VALUES = {
    "",
    "na",
    "n/a",
    "nan",
    "none",
    "null",
    "#n/a",
    "#na",
    "-",
    "--",
    "*",
    "**",
    "***",
    "~",
}

# ============================================================
# FILE CHECK
# ============================================================

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("=" * 100)
print("JOB CSV DECISION SCAN ONLY")
print("=" * 100)
print(f"File: {FILE}")
print(f"File size: {FILE.stat().st_size / (1024 ** 2):,.2f} MB")
print(f"Chunksize: {CHUNKSIZE:,}")
print(f"Duplicate key scan: {CHECK_DUPLICATE_KEYS}")
print()

# ============================================================
# ENCODING CHECK
# ============================================================

ENCODING = None

for enc in ["utf-8", "utf-8-sig", "latin1"]:
    try:
        pd.read_csv(FILE, nrows=0, encoding=enc)
        ENCODING = enc
        break
    except UnicodeDecodeError:
        continue

if ENCODING is None:
    raise ValueError("Could not detect encoding using utf-8, utf-8-sig, or latin1.")

print(f"Encoding used: {ENCODING}")
print()

# ============================================================
# HEADER ONLY
# ============================================================

header_df = pd.read_csv(FILE, nrows=0, encoding=ENCODING)
ALL_COLS = list(header_df.columns)

print("=" * 100)
print("BASIC FILE STRUCTURE")
print("=" * 100)
print(f"Total columns: {len(ALL_COLS):,}")
print()

for i, col in enumerate(ALL_COLS, start=1):
    print(f"{i:>3}. {col}")

print()

# ============================================================
# COLUMN EXISTENCE CHECK
# ============================================================

needed_cols = sorted(set(NUMERIC_ANALYSIS_COLS + TEXT_ANALYSIS_COLS + DUPLICATE_KEY_COLS))
missing_cols = [c for c in needed_cols if c not in ALL_COLS]

print("=" * 100)
print("COLUMN EXISTENCE CHECK")
print("=" * 100)

if missing_cols:
    print("WARNING: These expected columns are missing:")
    for c in missing_cols:
        print(f"- {c}")
else:
    print("All expected analysis columns exist.")

print()

# Keep only columns that exist
NUMERIC_ANALYSIS_COLS = [c for c in NUMERIC_ANALYSIS_COLS if c in ALL_COLS]
TEXT_ANALYSIS_COLS = [c for c in TEXT_ANALYSIS_COLS if c in ALL_COLS]
DUPLICATE_KEY_COLS = [c for c in DUPLICATE_KEY_COLS if c in ALL_COLS]

# ============================================================
# PREVIEW ONLY
# ============================================================

print("=" * 100)
print("FIRST 10 ROWS PREVIEW")
print("=" * 100)

preview_df = pd.read_csv(
    FILE,
    nrows=10,
    encoding=ENCODING,
    dtype="string",
    keep_default_na=False,
)

display(preview_df)

print()

# ============================================================
# SUMMARY COUNTERS
# ============================================================

total_rows = 0
chunk_count = 0

rows_by_year = Counter()
rows_by_data_level = Counter()
rows_by_year_data_level = Counter()
rows_by_source_workbook = Counter()
rows_by_source_file = Counter()
rows_by_source_sheet = Counter()

blank_counts = Counter()
na_like_counts = Counter()
dirty_symbol_counts = defaultdict(Counter)
whitespace_counts = Counter()

numeric_ok_counts = Counter()
numeric_bad_counts = Counter()
numeric_nonblank_counts = Counter()

text_blank_counts = Counter()
text_nonblank_counts = Counter()

# For duplicate key scan
seen_key_hashes = set()
duplicate_key_rows = 0
duplicate_key_counter = Counter()

start_time = time.time()

print("=" * 100)
print("SCANNING IN CHUNKS")
print("=" * 100)

reader = pd.read_csv(
    FILE,
    chunksize=CHUNKSIZE,
    encoding=ENCODING,
    dtype="string",
    keep_default_na=False,
    low_memory=False,
)

for chunk in reader:
    chunk_count += 1
    chunk_rows = len(chunk)
    total_rows += chunk_rows

    # --------------------------------------------------------
    # Strip only for checking, does not save or change file
    # --------------------------------------------------------
    for col in chunk.columns:
        chunk[col] = chunk[col].astype("string")

    # --------------------------------------------------------
    # Rows by year
    # --------------------------------------------------------
    if "year" in chunk.columns:
        year_s = chunk["year"].str.strip()
        rows_by_year.update(year_s.value_counts(dropna=False).to_dict())

    # --------------------------------------------------------
    # Rows by data_level
    # --------------------------------------------------------
    if "data_level" in chunk.columns:
        dl_s = chunk["data_level"].str.strip()
        rows_by_data_level.update(dl_s.value_counts(dropna=False).to_dict())

    # --------------------------------------------------------
    # Rows by year + data_level
    # --------------------------------------------------------
    if "year" in chunk.columns and "data_level" in chunk.columns:
        yd = (
            chunk[["year", "data_level"]]
            .fillna("")
            .astype("string")
            .apply(lambda x: x.str.strip())
        )
        yd_key = yd["year"] + " | " + yd["data_level"]
        rows_by_year_data_level.update(yd_key.value_counts(dropna=False).to_dict())

    # --------------------------------------------------------
    # Source counts
    # --------------------------------------------------------
    if "source_workbook" in chunk.columns:
        rows_by_source_workbook.update(
            chunk["source_workbook"].str.strip().value_counts(dropna=False).to_dict()
        )

    if "source_file" in chunk.columns:
        rows_by_source_file.update(
            chunk["source_file"].str.strip().value_counts(dropna=False).to_dict()
        )

    if "source_sheet" in chunk.columns:
        rows_by_source_sheet.update(
            chunk["source_sheet"].str.strip().value_counts(dropna=False).to_dict()
        )

    # --------------------------------------------------------
    # Text column blank checks
    # --------------------------------------------------------
    for col in TEXT_ANALYSIS_COLS:
        if col not in chunk.columns:
            continue

        s = chunk[col].astype("string")
        stripped = s.str.strip()
        lower = stripped.str.lower()

        blank_mask = stripped.eq("")
        text_blank_counts[col] += int(blank_mask.sum())
        text_nonblank_counts[col] += int((~blank_mask).sum())

        whitespace_counts[col] += int((s.ne(stripped) & ~blank_mask).sum())
        na_like_counts[col] += int(lower.isin(NA_LIKE_VALUES).sum())

    # --------------------------------------------------------
    # Numeric column dirty-value checks
    # --------------------------------------------------------
    for col in NUMERIC_ANALYSIS_COLS:
        if col not in chunk.columns:
            continue

        s = chunk[col].astype("string")
        stripped = s.str.strip()
        lower = stripped.str.lower()

        blank_mask = stripped.eq("")
        nonblank_mask = ~blank_mask

        blank_counts[col] += int(blank_mask.sum())
        na_like_counts[col] += int(lower.isin(NA_LIKE_VALUES).sum())
        whitespace_counts[col] += int((s.ne(stripped) & nonblank_mask).sum())
        numeric_nonblank_counts[col] += int(nonblank_mask.sum())

        for sym in DIRTY_SYMBOLS:
            dirty_symbol_counts[col][sym] += int(stripped.eq(sym).sum())

        numeric_test = (
            stripped
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        converted = pd.to_numeric(numeric_test[nonblank_mask], errors="coerce")

        numeric_ok_counts[col] += int(converted.notna().sum())
        numeric_bad_counts[col] += int(converted.isna().sum())

    # --------------------------------------------------------
    # Duplicate key scan
    # --------------------------------------------------------
    if CHECK_DUPLICATE_KEYS and len(DUPLICATE_KEY_COLS) > 0:
        key_df = (
            chunk[DUPLICATE_KEY_COLS]
            .fillna("")
            .astype("string")
            .apply(lambda x: x.str.strip())
        )

        key_hashes = pd.util.hash_pandas_object(key_df, index=False).values

        for h in key_hashes:
            if h in seen_key_hashes:
                duplicate_key_rows += 1
                duplicate_key_counter[h] += 1
            else:
                seen_key_hashes.add(h)

    elapsed = time.time() - start_time
    speed = total_rows / elapsed if elapsed > 0 else 0

    print(
        f"Chunk {chunk_count:,} done | "
        f"Rows read: {total_rows:,} | "
        f"Speed: {speed:,.0f} rows/sec | "
        f"Time: {elapsed:,.1f} sec",
        flush=True
    )

    del chunk
    gc.collect()

elapsed_total = time.time() - start_time

print()
print("=" * 100)
print("SCAN COMPLETE")
print("=" * 100)
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(ALL_COLS):,}")
print(f"Chunks scanned: {chunk_count:,}")
print(f"Total time: {elapsed_total:,.2f} seconds")

if CHECK_DUPLICATE_KEYS:
    print(f"Possible duplicate key rows: {duplicate_key_rows:,}")
    print(f"Unique key hashes seen: {len(seen_key_hashes):,}")

print()

# ============================================================
# TABLE 1: ROWS BY YEAR
# ============================================================

print("=" * 100)
print("ROWS BY YEAR")
print("=" * 100)

year_df = pd.DataFrame(
    rows_by_year.items(),
    columns=["year", "row_count"]
)

year_df["year_sort"] = pd.to_numeric(year_df["year"], errors="coerce")
year_df = year_df.sort_values(["year_sort", "year"]).drop(columns=["year_sort"])

display(year_df)

# ============================================================
# TABLE 2: ROWS BY DATA LEVEL
# ============================================================

print("=" * 100)
print("ROWS BY DATA_LEVEL")
print("=" * 100)

data_level_df = pd.DataFrame(
    rows_by_data_level.items(),
    columns=["data_level", "row_count"]
).sort_values("row_count", ascending=False)

display(data_level_df)

# ============================================================
# TABLE 3: YEAR + DATA LEVEL
# ============================================================

print("=" * 100)
print("ROWS BY YEAR + DATA_LEVEL")
print("=" * 100)

yd_rows = []

for key, count in rows_by_year_data_level.items():
    if " | " in key:
        year, data_level = key.split(" | ", 1)
    else:
        year, data_level = key, ""

    yd_rows.append({
        "year": year,
        "data_level": data_level,
        "row_count": count
    })

yd_df = pd.DataFrame(yd_rows)

if len(yd_df) > 0:
    yd_df["year_sort"] = pd.to_numeric(yd_df["year"], errors="coerce")
    yd_df = yd_df.sort_values(["year_sort", "data_level"]).drop(columns=["year_sort"])

display(yd_df)

# ============================================================
# TABLE 4: NUMERIC COLUMN DIRTY CHECK
# ============================================================

print("=" * 100)
print("NUMERIC ANALYSIS COLUMN CHECK")
print("=" * 100)

numeric_rows = []

for col in NUMERIC_ANALYSIS_COLS:
    nonblank = numeric_nonblank_counts[col]
    numeric_ok = numeric_ok_counts[col]
    numeric_bad = numeric_bad_counts[col]

    numeric_rows.append({
        "column": col,
        "total_rows": total_rows,
        "nonblank_count": nonblank,
        "blank_count": blank_counts[col],
        "na_like_count": na_like_counts[col],
        "numeric_ok_count": numeric_ok,
        "numeric_bad_count": numeric_bad,
        "numeric_ok_percent_of_nonblank": round((numeric_ok / nonblank * 100), 2) if nonblank else 0,
        "#": dirty_symbol_counts[col]["#"],
        "*": dirty_symbol_counts[col]["*"],
        "**": dirty_symbol_counts[col]["**"],
        "***": dirty_symbol_counts[col]["***"],
        "~": dirty_symbol_counts[col]["~"],
        "-": dirty_symbol_counts[col]["-"],
        "--": dirty_symbol_counts[col]["--"],
        "leading_trailing_space_count": whitespace_counts[col],
    })

numeric_check_df = pd.DataFrame(numeric_rows)

numeric_check_df = numeric_check_df.sort_values(
    by=["numeric_bad_count", "blank_count", "na_like_count"],
    ascending=False
).reset_index(drop=True)

display(numeric_check_df)

# ============================================================
# TABLE 5: TEXT COLUMN BLANK CHECK
# ============================================================

print("=" * 100)
print("TEXT / CATEGORY COLUMN CHECK")
print("=" * 100)

text_rows = []

for col in TEXT_ANALYSIS_COLS:
    text_rows.append({
        "column": col,
        "total_rows": total_rows,
        "nonblank_count": text_nonblank_counts[col],
        "blank_count": text_blank_counts[col],
        "blank_percent": round((text_blank_counts[col] / total_rows * 100), 2) if total_rows else 0,
        "na_like_count": na_like_counts[col],
        "leading_trailing_space_count": whitespace_counts[col],
    })

text_check_df = pd.DataFrame(text_rows)

text_check_df = text_check_df.sort_values(
    by=["blank_count", "na_like_count", "leading_trailing_space_count"],
    ascending=False
).reset_index(drop=True)

display(text_check_df)

# ============================================================
# TABLE 6: TOP SOURCE WORKBOOKS
# ============================================================

print("=" * 100)
print("TOP SOURCE WORKBOOKS")
print("=" * 100)

source_workbook_df = pd.DataFrame(
    rows_by_source_workbook.most_common(30),
    columns=["source_workbook", "row_count"]
)

display(source_workbook_df)

# ============================================================
# TABLE 7: TOP SOURCE FILES
# ============================================================

print("=" * 100)
print("TOP SOURCE FILES")
print("=" * 100)

source_file_df = pd.DataFrame(
    rows_by_source_file.most_common(30),
    columns=["source_file", "row_count"]
)

display(source_file_df)

# ============================================================
# TABLE 8: TOP SOURCE SHEETS
# ============================================================

print("=" * 100)
print("TOP SOURCE SHEETS")
print("=" * 100)

source_sheet_df = pd.DataFrame(
    rows_by_source_sheet.most_common(30),
    columns=["source_sheet", "row_count"]
)

display(source_sheet_df)

# ============================================================
# FINAL DECISION PRINT
# ============================================================

print("=" * 100)
print("FINAL CLEANING DECISION HELP")
print("=" * 100)

print(f"Rows: {total_rows:,}")
print(f"Columns: {len(ALL_COLS):,}")

if CHECK_DUPLICATE_KEYS:
    print(f"Possible duplicate key rows using {DUPLICATE_KEY_COLS}: {duplicate_key_rows:,}")

print()

# Numeric issue decision
bad_numeric_cols = numeric_check_df[
    (numeric_check_df["numeric_bad_count"] > 0) |
    (numeric_check_df["#"] > 0) |
    (numeric_check_df["*"] > 0) |
    (numeric_check_df["~"] > 0)
]["column"].tolist()

high_blank_text_cols = text_check_df[
    text_check_df["blank_percent"] >= 40
]["column"].tolist()

print("CLEANING NEEDED FOR ANALYSIS COLUMNS?")

if bad_numeric_cols:
    print("YES - numeric columns need light cleaning before math/chart:")
    for col in bad_numeric_cols:
        print(f"- {col}")
else:
    print("NO major numeric cleaning needed.")

print()

print("HIGH BLANK TEXT/CATEGORY COLUMNS:")
if high_blank_text_cols:
    for col in high_blank_text_cols:
        print(f"- {col}")
else:
    print("No text/category column has 40%+ blanks.")

print()

print("RECOMMENDATION:")
print("1. Do NOT drop rows just because blanks exist.")
print("2. Do NOT clean source columns unless needed.")
print("3. Before charts/math, convert #, *, **, ***, ~, blank in numeric columns to missing.")
print("4. Check duplicate keys before deleting duplicates.")
print("5. Keep source_workbook, source_file, source_sheet for tracking.")
print()
print("This scan did not clean or save anything.")

# Cleanning 1

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import time
import gc

# ============================================================
# CLEAN JOB CSV - NUMERIC ONLY
# MEMORY OPTIMIZED VERSION
#
# INPUT:
# ~/Downloads/all_2010-2025_job.csv
#
# OUTPUT:
# ~/Downloads/all_2010-2025_job_CLEANED_numeric_only_1.csv
#
# DOES:
# 1. Reads big CSV in chunks
# 2. Keeps all rows
# 3. Keeps all columns
# 4. Does NOT remove duplicates
# 5. Does NOT drop blank rows
# 6. Cleans numeric analysis columns only
# 7. Converts #, *, **, ***, ~, -, --, blank to missing
# 8. Removes comma, dollar sign, percent sign from numeric columns
# 9. Converts numeric columns to real numbers while writing
# 10. After saving, scans cleaned file to confirm
#
# MEMORY OPTIMIZATION:
# - chunksize reading
# - no full dataframe loaded
# - deletes each chunk after writing
# - only keeps small summary counters
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job.csv"

OUTPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

CHUNKSIZE = 100_000

# From your previous scan
EXPECTED_ROWS = 6_597_966
EXPECTED_COLS = 46

# Numeric columns that need cleaning before math/chart
NUMERIC_COLS = [
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_quotient",
    "pct_total",
    "pct_rpt",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
]

# Values to become blank/missing in numeric columns
BAD_NUMERIC_VALUES = {
    "",
    "na",
    "n/a",
    "nan",
    "none",
    "null",
    "#n/a",
    "#na",
    "#",
    "*",
    "**",
    "***",
    "~",
    "-",
    "--",
}

# Dirty symbols to confirm after save
DIRTY_SYMBOLS = ["#", "*", "**", "***", "~", "-", "--"]

# ============================================================
# SAFETY CHECK
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found:\n{INPUT_FILE}")

if OUTPUT_FILE.exists():
    print(f"Old output file already exists. Removing it first:\n{OUTPUT_FILE}")
    OUTPUT_FILE.unlink()

print("=" * 100)
print("CLEAN JOB CSV - NUMERIC ONLY")
print("=" * 100)
print(f"Input file : {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Input size : {INPUT_FILE.stat().st_size / (1024 ** 2):,.2f} MB")
print(f"Chunksize  : {CHUNKSIZE:,}")
print()

# ============================================================
# ENCODING CHECK
# ============================================================

ENCODING = None

for enc in ["utf-8", "utf-8-sig", "latin1"]:
    try:
        pd.read_csv(INPUT_FILE, nrows=0, encoding=enc)
        ENCODING = enc
        break
    except UnicodeDecodeError:
        continue

if ENCODING is None:
    raise ValueError("Could not detect encoding using utf-8, utf-8-sig, or latin1.")

print(f"Encoding used: {ENCODING}")
print()

# ============================================================
# HEADER CHECK
# ============================================================

header_df = pd.read_csv(INPUT_FILE, nrows=0, encoding=ENCODING)
ALL_COLS = list(header_df.columns)

missing_numeric_cols = [c for c in NUMERIC_COLS if c not in ALL_COLS]

print("=" * 100)
print("COLUMN CHECK")
print("=" * 100)
print(f"Columns found: {len(ALL_COLS):,}")

if missing_numeric_cols:
    print("WARNING: These numeric columns were not found and will be skipped:")
    for c in missing_numeric_cols:
        print(f"- {c}")
else:
    print("All numeric cleaning columns found.")

print()

NUMERIC_COLS = [c for c in NUMERIC_COLS if c in ALL_COLS]

# ============================================================
# CLEAN + SAVE IN CHUNKS
# ============================================================

print("=" * 100)
print("CLEANING AND WRITING FILE")
print("=" * 100)

start_time = time.time()

total_rows_written = 0
chunk_count = 0

before_bad_value_counts = defaultdict(Counter)
after_bad_value_counts = defaultdict(Counter)
numeric_bad_after_counts = Counter()
blank_after_counts = Counter()

reader = pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNKSIZE,
    encoding=ENCODING,
    dtype="string",
    keep_default_na=False,
    low_memory=False,
)

for chunk in reader:
    chunk_count += 1
    chunk_rows = len(chunk)

    # --------------------------------------------------------
    # Trim leading/trailing spaces for all columns
    # This does not remove inside spaces.
    # --------------------------------------------------------
    for col in chunk.columns:
        chunk[col] = chunk[col].astype("string").str.strip()

    # --------------------------------------------------------
    # Clean numeric columns only
    # --------------------------------------------------------
    for col in NUMERIC_COLS:
        s = chunk[col].astype("string").str.strip()

        # Count dirty values before cleaning
        for sym in DIRTY_SYMBOLS:
            before_bad_value_counts[col][sym] += int(s.eq(sym).sum())

        # Remove common numeric formatting
        s = (
            s
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )

        lower_s = s.str.lower()

        # Convert known bad/suppressed values to missing
        s = s.mask(lower_s.isin(BAD_NUMERIC_VALUES), pd.NA)

        # Convert to numeric
        numeric_s = pd.to_numeric(s, errors="coerce")

        # Count after cleaning
        after_as_string = numeric_s.astype("string").str.strip()

        blank_after_counts[col] += int(after_as_string.isna().sum())

        for sym in DIRTY_SYMBOLS:
            after_bad_value_counts[col][sym] += int(after_as_string.eq(sym).sum())

        # Bad after means nonblank but still cannot convert
        original_nonblank_after_clean = s.notna()
        numeric_bad_after_counts[col] += int((numeric_s.isna() & original_nonblank_after_clean).sum())

        # Put cleaned numeric column back
        chunk[col] = numeric_s

    # --------------------------------------------------------
    # Write chunk
    # --------------------------------------------------------
    write_header = chunk_count == 1
    write_mode = "w" if chunk_count == 1 else "a"

    chunk.to_csv(
        OUTPUT_FILE,
        mode=write_mode,
        header=write_header,
        index=False,
        encoding="utf-8",
        na_rep="",
        float_format="%.12g"
    )

    total_rows_written += chunk_rows

    elapsed = time.time() - start_time
    speed = total_rows_written / elapsed if elapsed > 0 else 0
    percent = (total_rows_written / EXPECTED_ROWS * 100) if EXPECTED_ROWS else 0

    print(
        f"Chunk {chunk_count:,} written | "
        f"Rows written: {total_rows_written:,} / {EXPECTED_ROWS:,} | "
        f"{percent:,.2f}% | "
        f"Speed: {speed:,.0f} rows/sec | "
        f"Time: {elapsed:,.1f} sec",
        flush=True
    )

    del chunk
    gc.collect()

clean_time = time.time() - start_time

print()
print("=" * 100)
print("CLEANING COMPLETE")
print("=" * 100)
print(f"Output file: {OUTPUT_FILE}")
print(f"Rows written: {total_rows_written:,}")
print(f"Output size: {OUTPUT_FILE.stat().st_size / (1024 ** 2):,.2f} MB")
print(f"Total clean/write time: {clean_time:,.2f} seconds")
print()

# ============================================================
# SUMMARY OF CLEANING ACTION
# ============================================================

print("=" * 100)
print("NUMERIC SYMBOLS REMOVED BEFORE WRITING")
print("=" * 100)

cleaning_summary_rows = []

for col in NUMERIC_COLS:
    row = {"column": col}

    total_dirty_before = 0
    total_dirty_after = 0

    for sym in DIRTY_SYMBOLS:
        before_count = before_bad_value_counts[col][sym]
        after_count = after_bad_value_counts[col][sym]

        row[f"before_{sym}"] = before_count
        row[f"after_{sym}"] = after_count

        total_dirty_before += before_count
        total_dirty_after += after_count

    row["total_dirty_before"] = total_dirty_before
    row["total_dirty_after"] = total_dirty_after
    row["numeric_bad_after_cleaning"] = numeric_bad_after_counts[col]
    row["blank_or_missing_after_cleaning"] = blank_after_counts[col]

    cleaning_summary_rows.append(row)

cleaning_summary_df = pd.DataFrame(cleaning_summary_rows)

cleaning_summary_df = cleaning_summary_df.sort_values(
    by=["total_dirty_before", "blank_or_missing_after_cleaning"],
    ascending=False
).reset_index(drop=True)

display(cleaning_summary_df)

# ============================================================
# SCAN CLEANED FILE AFTER COMPLETE
# ============================================================

print("=" * 100)
print("POST-CLEAN SCAN OF NEW FILE")
print("=" * 100)

post_start = time.time()

post_total_rows = 0
post_chunk_count = 0
post_cols = None

post_dirty_counts = defaultdict(Counter)
post_numeric_ok_counts = Counter()
post_numeric_bad_counts = Counter()
post_numeric_nonblank_counts = Counter()
post_blank_counts = Counter()

rows_by_year = Counter()

post_reader = pd.read_csv(
    OUTPUT_FILE,
    chunksize=CHUNKSIZE,
    encoding="utf-8",
    dtype="string",
    keep_default_na=False,
    low_memory=False,
)

for chunk in post_reader:
    post_chunk_count += 1

    if post_cols is None:
        post_cols = list(chunk.columns)

    post_total_rows += len(chunk)

    if "year" in chunk.columns:
        rows_by_year.update(chunk["year"].astype("string").str.strip().value_counts().to_dict())

    for col in NUMERIC_COLS:
        if col not in chunk.columns:
            continue

        s = chunk[col].astype("string").str.strip()
        blank_mask = s.eq("")
        nonblank_mask = ~blank_mask

        post_blank_counts[col] += int(blank_mask.sum())
        post_numeric_nonblank_counts[col] += int(nonblank_mask.sum())

        for sym in DIRTY_SYMBOLS:
            post_dirty_counts[col][sym] += int(s.eq(sym).sum())

        numeric_test = (
            s
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        converted = pd.to_numeric(numeric_test[nonblank_mask], errors="coerce")

        post_numeric_ok_counts[col] += int(converted.notna().sum())
        post_numeric_bad_counts[col] += int(converted.isna().sum())

    elapsed = time.time() - post_start
    speed = post_total_rows / elapsed if elapsed > 0 else 0
    percent = (post_total_rows / EXPECTED_ROWS * 100) if EXPECTED_ROWS else 0

    print(
        f"Post-scan chunk {post_chunk_count:,} done | "
        f"Rows scanned: {post_total_rows:,} / {EXPECTED_ROWS:,} | "
        f"{percent:,.2f}% | "
        f"Speed: {speed:,.0f} rows/sec | "
        f"Time: {elapsed:,.1f} sec",
        flush=True
    )

    del chunk
    gc.collect()

post_time = time.time() - post_start

print()
print("=" * 100)
print("POST-CLEAN SCAN COMPLETE")
print("=" * 100)
print(f"Rows scanned in cleaned file: {post_total_rows:,}")
print(f"Columns in cleaned file: {len(post_cols):,}")
print(f"Post-scan time: {post_time:,.2f} seconds")
print()

# ============================================================
# POST-CLEAN NUMERIC CHECK TABLE
# ============================================================

print("=" * 100)
print("POST-CLEAN NUMERIC CHECK")
print("=" * 100)

post_rows = []

for col in NUMERIC_COLS:
    total_dirty_after_scan = sum(post_dirty_counts[col][sym] for sym in DIRTY_SYMBOLS)

    post_rows.append({
        "column": col,
        "nonblank_count": post_numeric_nonblank_counts[col],
        "blank_count": post_blank_counts[col],
        "numeric_ok_count": post_numeric_ok_counts[col],
        "numeric_bad_count": post_numeric_bad_counts[col],
        "dirty_symbol_count_after": total_dirty_after_scan,
        "#": post_dirty_counts[col]["#"],
        "*": post_dirty_counts[col]["*"],
        "**": post_dirty_counts[col]["**"],
        "***": post_dirty_counts[col]["***"],
        "~": post_dirty_counts[col]["~"],
        "-": post_dirty_counts[col]["-"],
        "--": post_dirty_counts[col]["--"],
    })

post_numeric_check_df = pd.DataFrame(post_rows)

post_numeric_check_df = post_numeric_check_df.sort_values(
    by=["dirty_symbol_count_after", "numeric_bad_count", "blank_count"],
    ascending=False
).reset_index(drop=True)

display(post_numeric_check_df)

# ============================================================
# ROWS BY YEAR CHECK
# ============================================================

print("=" * 100)
print("ROWS BY YEAR IN CLEANED FILE")
print("=" * 100)

year_df = pd.DataFrame(
    rows_by_year.items(),
    columns=["year", "row_count"]
)

year_df["year_sort"] = pd.to_numeric(year_df["year"], errors="coerce")
year_df = year_df.sort_values(["year_sort", "year"]).drop(columns=["year_sort"])

display(year_df)

# ============================================================
# FINAL PASS / WARNING DECISION
# ============================================================

print("=" * 100)
print("FINAL RESULT")
print("=" * 100)

row_match = post_total_rows == EXPECTED_ROWS
col_match = len(post_cols) == EXPECTED_COLS

total_dirty_left = int(post_numeric_check_df["dirty_symbol_count_after"].sum())
total_numeric_bad_left = int(post_numeric_check_df["numeric_bad_count"].sum())

print(f"Expected rows: {EXPECTED_ROWS:,}")
print(f"Cleaned rows : {post_total_rows:,}")
print(f"Rows match   : {row_match}")
print()

print(f"Expected columns: {EXPECTED_COLS:,}")
print(f"Cleaned columns : {len(post_cols):,}")
print(f"Columns match   : {col_match}")
print()

print(f"Dirty numeric symbols left: {total_dirty_left:,}")
print(f"Nonblank numeric values that still fail numeric conversion: {total_numeric_bad_left:,}")
print()

if row_match and col_match and total_dirty_left == 0 and total_numeric_bad_left == 0:
    print("PASS: Cleaned numeric-only file is ready for analysis.")
else:
    print("WARNING: Check the post-clean numeric table above before analysis.")

print()
print(f"Saved cleaned file here:\n{OUTPUT_FILE}")
print()
print("Reminder: This code did NOT drop rows and did NOT delete duplicates.")

# Read column new file

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# READ COLUMNS ONLY
# MEMORY OPTIMIZED
#
# NO CLEANING
# NO SAVE
# NO FULL FILE LOAD
# ONLY READ HEADER + SMALL PREVIEW
# ============================================================

FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

start = time.time()

print("=" * 100)
print("FILE CHECK")
print("=" * 100)

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print(f"File found: {FILE}")
print(f"File size : {FILE.stat().st_size / (1024**2):,.2f} MB")

print("\n" + "=" * 100)
print("READING COLUMNS ONLY")
print("=" * 100)

# Read only header, not data
header_df = pd.read_csv(FILE, nrows=0, encoding="utf-8")

columns = list(header_df.columns)

column_table = pd.DataFrame({
    "column_number": range(1, len(columns) + 1),
    "column_name": columns
})

print(f"Columns count: {len(columns)}")

print("\n" + "=" * 100)
print("ALL COLUMNS")
print("=" * 100)
print(column_table.to_string(index=False))

print("\n" + "=" * 100)
print("PREVIEW FIRST 5 ROWS ONLY")
print("=" * 100)

preview_df = pd.read_csv(FILE, nrows=5, encoding="utf-8", low_memory=False)
print(preview_df.to_string(index=False))

end = time.time()

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)
print(f"Read column time: {end - start:.2f} seconds")
print("This code did NOT clean, save, drop rows, or load the full file.")